# Rumble-2-Flash — 45-language speech-to-text on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/Rumble-2-Flash-noisy-speech-to-text.ipynb)

[`Reson8/Rumble-2-Flash`](https://huggingface.co/Reson8/Rumble-2-Flash) is Reson8's **~0.51B** (508.6M) Qwen3-ASR speech recognizer. [Apache-2.0](https://huggingface.co/Reson8/Rumble-2-Flash), public (not gated). Hub: `transformers` + `safetensors`, architecture `Qwen3ASRForConditionalGeneration` (`model_type: qwen3_asr`). The card lists **45 languages** and documents native Hugging Face Transformers and vLLM. `model.safetensors` is **1.0 GB** (saved as bfloat16). The repo also ships an optional CTC head (`ctc/ctc_head.safetensors`, `ctc_head.py`); this notebook uses the card's Transformers decoder path, not CTC.

**Why this model.** The card's Transformers usage is `AutoProcessor` + `AutoModelForMultimodalLM` with `processor.apply_transcription_request(...)`. On the [Open ASR Leaderboard](https://huggingface.co/spaces/hf-audio/open_asr_leaderboard?version=1.16.1) (release 18-11-2026, v1.16.1, H200 + vLLM) the card reports English average WER **6.60** (RTFx 6338). Those are the card's numbers, not a T4 measurement from this notebook.

See [Use cases](https://github.com/donvito/notebooks/blob/main/docs/Rumble-2-Flash-noisy-speech-to-text.md) for when this notebook is a good fit and when it is not.

**Licence:** Apache-2.0.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. The free T4 has ~16 GB of VRAM and **no native bfloat16**. Saved weights are BF16; this notebook loads **float16**. `config.json` records `transformers_version: 5.21.0.dev0`. The card install is `uv pip install torch transformers accelerate` — **do not reinstall torch** (Colab already has a CUDA wheel). No `qwen-asr` package: that pin is for the sibling [cbx-healthcare](cbx-healthcare-speech-to-text-0.6B-medical-asr.ipynb) LoRA on `Qwen/Qwen3-ASR-0.6B`. Colab already ships Transformers 5.x, so this notebook does **not** pin 4.x or `huggingface_hub<1`.

**Not in this notebook:** vLLM serving, the optional CTC head, or training.

**Links:** [model card](https://huggingface.co/Reson8/Rumble-2-Flash) · [announcement](https://x.com/raoulritterr/status/2123053360773271563) · [use cases](https://github.com/donvito/notebooks/blob/main/docs/Rumble-2-Flash-noisy-speech-to-text.md) · [Open ASR Leaderboard](https://huggingface.co/spaces/hf-audio/open_asr_leaderboard?version=1.16.1)


## 0. Check the GPU

Confirm CUDA is available, print the device name and free VRAM. Need a T4 (or any CUDA GPU). A restart after the install cell drops this state; section 2 prints it again before load.


In [ ]:
import os
import shutil

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

import torch

print("torch.cuda.is_available():", torch.cuda.is_available())
print("torch", torch.__version__, "| cuda", torch.version.cuda)

if torch.cuda.is_available():
    free_b, total_b = torch.cuda.mem_get_info()
    major, minor = torch.cuda.get_device_capability()
    print("GPU name:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
    print(f"VRAM free / total: {free_b / 1024**3:.1f} / {total_b / 1024**3:.1f} GB")
    if major < 8:
        print("No native bfloat16 on this GPU; load will use float16.")
else:
    print("GPU name: none")
    print("VRAM free: n/a")
    print("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

print("nvidia-smi:", shutil.which("nvidia-smi") or "not on PATH")


## 1. Install (keep Colab's torch)

The [model card](https://huggingface.co/Reson8/Rumble-2-Flash) install is:

```bash
uv pip install torch transformers accelerate
```

`config.json` was saved with `transformers_version: 5.21.0.dev0`. Qwen3-ASR (`AutoModelForMultimodalLM`, `apply_transcription_request`) is native from Transformers **5.13**. This notebook pins **`transformers>=5.21.0`** to match the checkpoint, plus `accelerate` for `device_map="auto"`. `apply_chat_template` needs **`jinja2>=3.1.6`**. `librosa` and `soundfile` are only for the phone-quality resample demo.

**Do not reinstall torch** — Colab already ships a CUDA wheel; forcing a reinstall often drops GPU support. Use **`%pip`** (not `python -I -m pip`). Isolated mode drops Colab's site-packages and breaks the preinstalled torch.

**Sibling notebook note.** [cbx-healthcare](cbx-healthcare-speech-to-text-0.6B-medical-asr.ipynb) (Qwen3-ASR-0.6B LoRA) had to pin `transformers==4.57.6`, uninstall Colab `torchao` (peft 0.21.2 clash), and pin `huggingface_hub<1`. This card does **not** use `qwen-asr` or peft, and Colab already ships Transformers 5.x, so those 4.x pins would break this load. The install cell still **restarts the session** so the upgraded transformers is what gets imported.

**Do not import the model in this cell.** Colab has already imported its preinstalled transformers. After reconnect, skip this cell and continue from section 2.


In [ ]:
import importlib.metadata as md
import os
import signal
import sys
from pathlib import Path

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

INSTALL_TAG = "tf>=5.21.0-rumble2flash"
MARKER = (
    Path("/content/.rumble2flash_nb_install_ok")
    if Path("/content").is_dir()
    else Path(".rumble2flash_nb_install_ok")
)
already_restarted = MARKER.exists() and MARKER.read_text().strip() == INSTALL_TAG

print("python", sys.version.split()[0])
try:
    import torch

    print("torch (keep Colab's wheel)", torch.__version__, "| cuda", torch.version.cuda)
except ImportError:
    print("torch not imported yet")

# Public weights: skip Colab userdata.get("HF_TOKEN") popups.
try:
    import huggingface_hub.utils._auth as _hf_auth

    _hf_auth._IS_GOOGLE_COLAB_CHECKED = True
    _hf_auth._GOOGLE_COLAB_SECRET = None
except Exception:
    pass

# Card: torch transformers accelerate. Keep Colab's CUDA torch.
# transformers>=5.21.0 matches config.json (5.21.0.dev0). Do not pin 4.x.
get_ipython().run_line_magic(
    "pip",
    'install -q "transformers>=5.21.0" accelerate librosa soundfile "jinja2>=3.1.6"',
)

print()
print("transformers", md.version("transformers"))
print("accelerate", md.version("accelerate"))
print("librosa", md.version("librosa"))
print("huggingface_hub", md.version("huggingface_hub"))
print("jinja2", md.version("jinja2"))

MARKER.write_text(INSTALL_TAG)
if already_restarted:
    print("Install marker already set; not restarting again. Continue from section 2.")
else:
    print()
    print("=" * 72)
    print("Restarting the session so transformers>=5.21.0 is imported.")
    print("After reconnect: skip this cell and run from section 2 (runtime check).")
    print("A 'session crashed' or 'Operation cancelled' message here is the restart,")
    print("not a failed install — pip already finished (versions printed above).")
    print("=" * 72)
    os.kill(os.getpid(), signal.SIGKILL)


> The install cell **restarts the session for you**. After reconnect, skip it and continue from section 2.
> A "session crashed" message after install is the restart, not a failed pip. If you restarted by hand instead: *Runtime > Restart session*, then this next cell.


## 2. Runtime check after restart

Confirm `transformers>=5.21.0`, CUDA, device name, and free VRAM. T4 (`sm_75`) has no native bfloat16 — **float16** is what this notebook loads.


In [ ]:
import os
import time
from pathlib import Path

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
try:
    import huggingface_hub.utils._auth as _hf_auth

    _hf_auth._IS_GOOGLE_COLAB_CHECKED = True
    _hf_auth._GOOGLE_COLAB_SECRET = None
except Exception:
    pass

import torch
import transformers
from packaging.version import Version

assert Version(transformers.__version__) >= Version("5.21.0"), (
    f"transformers {transformers.__version__} is loaded; Rumble-2-Flash needs 5.21+ "
    "(config.json transformers_version 5.21.0.dev0). Run the install cell, then "
    "Runtime > Restart session."
)

print("torch.cuda.is_available():", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

major, minor = torch.cuda.get_device_capability()
DEVICE = "cuda"
# T4 (sm_75) has no native bfloat16. Checkpoint dtype is bfloat16.
DTYPE = torch.float16 if major < 8 else torch.bfloat16
MODEL_ID = "Reson8/Rumble-2-Flash"
SR = 16_000
CHUNK_SECONDS = 30  # preprocessor_config.json chunk_length / n_samples=480000

free_b, total_b = torch.cuda.mem_get_info()
print("transformers", transformers.__version__)
print("torch", torch.__version__)
print("GPU name:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
print(f"dtype: {DTYPE}  (saved weights are bfloat16; T4 has no native bf16)")
print(f"VRAM free / total: {free_b / 1024**3:.1f} / {total_b / 1024**3:.1f} GB")
if major < 8:
    print("No native bfloat16 on this GPU — staying on float16.")


def print_vram(label=""):
    suffix = f"  ({label})" if label else ""
    allocated = torch.cuda.memory_allocated() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    free_now, total_now = torch.cuda.mem_get_info()
    print(
        f"VRAM allocated: {allocated:.2f} GB | peak: {peak:.2f} GB | "
        f"free: {free_now / 1024**3:.1f}/{total_now / 1024**3:.1f} GB{suffix}"
    )


## 3. Load the model

The card's call is:

```python
from transformers import AutoProcessor, AutoModelForMultimodalLM

model_id = "Reson8/Rumble-2-Flash"
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForMultimodalLM.from_pretrained(model_id, device_map="auto")
```

We only add `dtype=torch.float16` on T4 (the card's default would follow the BF16 checkpoint). First run downloads **1.0 GB** (`model.safetensors`). No `trust_remote_code` — `config.json` names native `Qwen3ASRForConditionalGeneration`.


In [ ]:
from transformers import AutoModelForMultimodalLM, AutoProcessor

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

t0 = time.perf_counter()
processor = AutoProcessor.from_pretrained(MODEL_ID, token=False)
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    dtype=DTYPE,
    token=False,
)
torch.cuda.synchronize()
print(f"loaded {MODEL_ID} in {time.perf_counter() - t0:.1f}s")
print("class:", type(model).__name__)
print("dtype:", next(model.parameters()).dtype, "| device:", next(model.parameters()).device)
n_params = sum(p.numel() for p in model.parameters())
print(f"parameters: {n_params / 1e6:.1f}M  (Hub safetensors: 508.6M)")
print_vram("after load")


def transcribe(audio, language=None, max_new_tokens=512, **kwargs):
    """Card usage: apply_transcription_request -> generate -> decode transcription_only.

    language=None detects automatically (use parsed output). English name such as
    \"English\" forces the language. Extra kwargs (digits, filler, verbatim) go to
    apply_chat_template, as on the card.
    """
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    inputs = processor.apply_transcription_request(
        audio=audio,
        language=language,
        **kwargs,
    ).to(model.device, dtype=model.dtype)
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens)
    generated = output[:, inputs["input_ids"].shape[-1] :]
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    text = processor.decode(generated, return_format="transcription_only")[0]
    parsed = processor.decode(generated, return_format="parsed")[0]
    peak = (
        torch.cuda.max_memory_allocated() / 1024**3
        if torch.cuda.is_available()
        else None
    )
    return {"text": text, "parsed": parsed, "elapsed": elapsed, "peak_gb": peak}


def report(out, seconds, label):
    print(f"{label}: {out['text']}")
    print(f"parsed: {out['parsed']}")
    rtf = out["elapsed"] / seconds if seconds else float("nan")
    peak = f"{out['peak_gb']:.2f} GB" if out["peak_gb"] is not None else "n/a"
    print(
        f"audio: {seconds:.2f}s  |  wall time: {out['elapsed']:.2f}s  |  "
        f"RTF {rtf:.3f}x  |  peak {peak}"
    )
    print_vram(label)


## 4. Demo 1 — clean English sample

Public LibriSpeech-style clip [`librispeech_mr_quilter.wav`](https://huggingface.co/datasets/hf-internal-testing/dummy-audio-samples) from Hugging Face's dummy audio set (the same family as the Qwen3-ASR docs). Forced `language="English"` as in the card. When language is forced, `return_format="parsed"` may show `language: None` because that tag was prefilled and the model only emits the transcript.

If that download fails, the cell tries the card's own example URL, then a Hugging Face speech sample, then a short generated tone so later cells still run.


In [ ]:
import urllib.request
from pathlib import Path

import librosa
import numpy as np
import soundfile as sf
from IPython.display import Audio, display

SAMPLE_DIR = Path("rumble2_samples")
SAMPLE_DIR.mkdir(exist_ok=True)
UA = {"User-Agent": "donvito-notebooks/1.0 (educational Colab demo)"}

DUMMY = (
    "https://huggingface.co/datasets/hf-internal-testing/dummy-audio-samples/resolve/main"
)
EN_URL = f"{DUMMY}/librispeech_mr_quilter.wav"
CARD_URL = f"{DUMMY}/bcn_weather.mp3"
HF_SPEECH_SAMPLE = "https://cdn-media.huggingface.co/speech_samples/sample1.flac"
ES_URL = f"{DUMMY}/fleur_es_sample.wav"
ZH_URL = f"{DUMMY}/mandarin_voxcpm_zh.wav"
LONG_URL = f"{DUMMY}/obama_first_45_secs.mp3"


def _urlretrieve(url, path, timeout=120):
    req = urllib.request.Request(url, headers=UA)
    with urllib.request.urlopen(req, timeout=timeout) as resp, open(path, "wb") as f:
        f.write(resp.read())
    return Path(path)


def load_mono(path, sr=SR):
    y, _ = librosa.load(path, sr=sr, mono=True)
    return np.asarray(y, dtype=np.float32)


def write_wav(path, audio, sr=SR):
    audio = np.asarray(audio, dtype=np.float32)
    if audio.ndim > 1:
        audio = audio.mean(axis=-1)
    sf.write(path, audio, int(sr))
    return Path(path), len(audio) / float(sr)


def download_first(urls_and_names):
    last_err = None
    for url, name in urls_and_names:
        dest = SAMPLE_DIR / name
        try:
            if not dest.exists():
                print("Downloading", url)
                _urlretrieve(url, dest)
            y = load_mono(dest)
            print(f"loaded {dest.name}: {len(y) / SR:.2f}s")
            return dest, y
        except Exception as exc:
            last_err = exc
            print(f"failed {url}: {exc!r}")
    print(f"All downloads failed ({last_err!r}); using 3 s of silence so later cells run.")
    y = np.zeros(SR * 3, dtype=np.float32)
    dest, _ = write_wav(SAMPLE_DIR / "silence.wav", y)
    return dest, y


en_src, en_audio = download_first(
    [
        (EN_URL, "librispeech_mr_quilter.wav"),
        (f"{DUMMY}/fleurs_en_us_test_0346.wav", "fleurs_en_us_test_0346.wav"),
        (HF_SPEECH_SAMPLE, "hf-speech-sample1.flac"),
    ]
)
en_path, en_seconds = write_wav(SAMPLE_DIR / "demo1-clean-en.wav", en_audio)
print(f"clean English: {en_path.name} ({en_seconds:.2f}s)")
display(Audio(str(en_path)))

en_out = transcribe(str(en_path), language="English")
print()
report(en_out, en_seconds, "clean English")


## 5. Demo 2 — phone-quality version of the same clip

Same utterance as Demo 1, with **added noise** and an **8 kHz resample** (then back to 16 kHz) to mimic a narrowband phone call. This is a synthetic degradation for the notebook, not a card benchmark. The card does publish URGENT24 WER **10.25** and AMI WER **9.24** on the English leaderboard table (H200 + vLLM); those are not this clip.


In [ ]:
rng = np.random.default_rng(0)
noise_std = 0.02 * (float(np.sqrt(np.mean(en_audio**2))) + 1e-3)
noisy = np.clip(en_audio + rng.normal(0.0, noise_std, size=en_audio.shape).astype(np.float32), -1.0, 1.0)
y8 = librosa.resample(noisy, orig_sr=SR, target_sr=8_000)
phone = librosa.resample(y8, orig_sr=8_000, target_sr=SR)
# resample can change length by a few samples
n = min(len(en_audio), len(phone))
phone = phone[:n]

phone_path, phone_seconds = write_wav(SAMPLE_DIR / "demo2-phone-en.wav", phone)
print(
    f"phone-quality: noise_std={noise_std:.4f}, 16 kHz -> 8 kHz -> 16 kHz, "
    f"{phone_seconds:.2f}s"
)
display(Audio(str(phone_path)))

phone_out = transcribe(str(phone_path), language="English")
print()
print("clean: ", en_out["text"])
report(phone_out, phone_seconds, "phone-quality English")


## 6. Demo 3 — non-English clip (auto-detect)

Spanish FLEURS excerpt [`fleur_es_sample.wav`](https://huggingface.co/datasets/hf-internal-testing/dummy-audio-samples). **No `language=` argument** — the card: omit it to detect automatically, and use `return_format="parsed"` for the language tag. Fallback: the card's own `bcn_weather.mp3` example URL, then Mandarin `mandarin_voxcpm_zh.wav`.


In [ ]:
xx_src, xx_audio = download_first(
    [
        (ES_URL, "fleur_es_sample.wav"),
        (CARD_URL, "bcn_weather.mp3"),
        (ZH_URL, "mandarin_voxcpm_zh.wav"),
    ]
)
xx_path, xx_seconds = write_wav(SAMPLE_DIR / "demo3-non-english.wav", xx_audio)
print(f"non-English source file: {xx_src.name} ({xx_seconds:.2f}s)")
display(Audio(str(xx_path)))

xx_out = transcribe(str(xx_path), language=None)  # auto-detect
print()
report(xx_out, xx_seconds, "non-English auto-detect")


## 7. Demo 4 — longer clip

[`obama_first_45_secs.mp3`](https://huggingface.co/datasets/hf-internal-testing/dummy-audio-samples) (~45 s). The feature extractor lists `chunk_length: 30` / `n_samples: 480000` (30 s at 16 kHz). The encoder uses 8 s attention windows (`n_window_infer: 800`), so longer files are expected to work; if generate fails on the full clip, the cell falls back to 30 s chunks and concatenates the text. Fallback: concatenate the clean English clip a few times.

This is a T4 wall-clock check, not the card's H200 RTFx 6338.


In [ ]:
long_src, long_audio = download_first(
    [
        (LONG_URL, "obama_first_45_secs.mp3"),
        (f"{DUMMY}/Ch6Ae9DT6Ko_00-04-03_00-04-31.wav", "yt-28s.wav"),
    ]
)
if long_src.name == "silence.wav" or len(long_audio) < 5 * SR:
    print("Long-clip download was short; concatenating the English demo instead.")
    gap = np.zeros(int(0.25 * SR), dtype=np.float32)
    long_audio = np.concatenate([en_audio, gap, en_audio, gap, en_audio, gap, en_audio])

long_path, long_seconds = write_wav(SAMPLE_DIR / "demo4-long.wav", long_audio)
print(f"long clip: {long_path.name} ({long_seconds:.2f}s)")
display(Audio(str(long_path)))

max_samples = int(CHUNK_SECONDS * SR)
try:
    long_out = transcribe(str(long_path), language="English")
    print()
    report(long_out, long_seconds, "long clip (full file)")
except Exception as exc:
    print(f"Full-file generate failed ({type(exc).__name__}: {exc}); chunking at {CHUNK_SECONDS}s.")
    texts = []
    elapsed = 0.0
    peak = 0.0
    for i, start in enumerate(range(0, len(long_audio), max_samples)):
        chunk = long_audio[start : start + max_samples]
        if len(chunk) < SR:
            continue
        chunk_path, chunk_seconds = write_wav(SAMPLE_DIR / f"demo4-chunk-{i:02d}.wav", chunk)
        out = transcribe(str(chunk_path), language="English")
        texts.append(out["text"])
        elapsed += out["elapsed"]
        if out["peak_gb"]:
            peak = max(peak, out["peak_gb"])
        print(f"  chunk {i}: {chunk_seconds:.1f}s -> {out['text'][:80]!r}...")
    joined = " ".join(texts)
    print()
    print("long clip (chunked):", joined)
    rtf = elapsed / long_seconds if long_seconds else float("nan")
    print(
        f"audio: {long_seconds:.2f}s  |  wall time: {elapsed:.2f}s  |  "
        f"RTF {rtf:.3f}x  |  peak {peak:.2f} GB"
    )
    print_vram("long clip chunked")


## 8. Caveats and cleanup

**What this notebook is for.** The card's Transformers ASR API on a free T4: clean English, a synthetic phone-quality version of the same clip, a non-English auto-detect clip, and a longer public file. Timing and `torch.cuda.max_memory_allocated` are printed per demo.

**What it is not.** vLLM serving, the optional CTC head (`ctc_head.py`: much less accurate than the decoder), or a leaderboard rerun. The card's English average WER 6.60 / RTFx 6338 is from an H200 + vLLM run.

**Related notebooks:** [cbx-healthcare-speech-to-text-0.6B](cbx-healthcare-speech-to-text-0.6B-medical-asr.ipynb) (Qwen3-ASR-0.6B LoRA, medical English), [Fluid Speech Pro](Fluid-Speech-Pro-multilingual-asr.ipynb) (25 European languages), [Nemotron-Labs Unified ASR 0.6B](Nemotron-Labs-Unified-ASR-0.6B-multilingual.ipynb), [Granite Speech 5.0 2B](granite-speech-5.0-2b-asr-translation.ipynb), [oruk-loop-153M](oruk-loop-153M-looped-asr.ipynb), [Whisper](AIBackends-Whisper-audio-transcription.ipynb).

Free the GPU before you open another notebook:


In [ ]:
import gc

print_vram("before cleanup")
print(f"peak VRAM this session: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
del model, processor
gc.collect()
torch.cuda.empty_cache()
print_vram("after cleanup")
